# Fundraising → Ballot Mentions as % of STV Threshold: District Explorer

## Question

**How does fundraising relate to candidate support after scaling mentions by each district's STV election threshold?**

This notebook is the threshold-normalized companion to the original weekly-slide Notebook 13b.

The only substantive change to the outcome is:

`mentions_pct_threshold = 100 × mentions / stv_threshold`

So:

- **100%** = mentions equal the district threshold;
- **150%** = mentions equal 1.5 times the threshold;
- **80%** = mentions equal 80% of the threshold.

This is especially useful when comparing candidates across districts with different thresholds.


## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from IPython.display import display, clear_output
import ipywidgets as widgets


# ---------------------------------------------------------------
# Find the repository root
# ---------------------------------------------------------------

cwd = Path.cwd().resolve()

for folder in [cwd, *cwd.parents]:
    if (folder / "pyproject.toml").exists():
        ROOT = folder
        break
else:
    raise FileNotFoundError("Could not find the repository root.")


YEAR = 2024

print("ROOT:", ROOT)


## 2. Load the current candidate-level table and build the normalized support measure

We use the same canonical finance × ballot table as the raw-mentions notebook.

No new CSV is created. The threshold-normalized variables are derived directly in the notebook.


In [ ]:
data_path = (
    ROOT
    / "data"
    / "processed"
    / "finance_vs_ballot_support"
    / str(YEAR)
    / "candidate_finance_ballot_analysis_2024.csv"
)


analysis = pd.read_csv(data_path)


# Keep the same variable name used in the original weekly notebook.
analysis["fundraising"] = analysis["total_amount"]


plot_data = analysis.dropna(
    subset=[
        "fundraising",
        "mentions",
        "stv_threshold",
    ]
).copy()


print("Candidates with matched fundraising:", len(plot_data))
print()
print("Candidates by district:")

display(
    plot_data["district"]
    .value_counts()
    .sort_index()
    .rename("candidates")
    .to_frame()
)


print()
print("STV threshold values by district:")

display(
    plot_data.groupby("district")["stv_threshold"]
    .agg(["min", "max", "nunique"])
)

# Support relative to each district's STV threshold.
plot_data["mentions_pct_threshold"] = (
    100
    * plot_data["mentions"]
    / plot_data["stv_threshold"]
)

plot_data["first_place_pct_threshold"] = (
    100
    * plot_data["first_place_votes"]
    / plot_data["stv_threshold"]
)


## 3. A tiny helper: correlation and R²

For each plot we use:

`mentions_pct_threshold = a + b × fundraising`


In [ ]:
def relationship_stats(data):
    # Pearson correlation between fundraising and mentions as % of threshold.
    correlation = data["fundraising"].corr(
        data["mentions_pct_threshold"]
    )

    # For one-predictor OLS with an intercept:
    r_squared = correlation ** 2

    return correlation, r_squared


## 4. Overall pattern: fundraising and mentions as % of threshold

Unlike within-district normalization, the pooled all-district relationship can change because each district has a different threshold.


In [ ]:
overall_r, overall_r2 = relationship_stats(
    plot_data
)


fig_overall = px.scatter(
    plot_data,
    x="fundraising",
    y="mentions_pct_threshold",
    hover_name="canonical_candidate",
    hover_data={
        "district": True,
        "fundraising": ":$,.0f",
        "mentions": ":,.0f",
        "stv_threshold": ":,.0f",
        "mentions_pct_threshold": ":.1f",
        "first_place_votes": ":,.0f",
        "first_place_pct_threshold": ":.1f",
        "is_viable": True,
    },
    trendline="ols",
    template="plotly_white",
    labels={
        "fundraising": "Total fundraising ($)",
        "mentions_pct_threshold": "Ballot mentions (% of STV threshold)",
    },
    title=(
        "2024 Portland City Council: fundraising and "
        "mentions relative to district threshold"
        f"<br><sup>Pearson r = {overall_r:.2f} | "
        f"R² = {overall_r2:.2f} | "
        f"n = {len(plot_data)}</sup>"
    ),
)


fig_overall.update_traces(
    marker={
        "size": 9,
        "opacity": 0.75,
    }
)


fig_overall.add_hline(
    y=100,
    line_dash="dash",
    line_color="gray",
    annotation_text="100% = district STV threshold",
)


fig_overall.show()


## 5. Compare the four districts

In [ ]:
district_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = plot_data[
        plot_data["district"].eq(district)
    ]

    r, r2 = relationship_stats(
        district_data
    )

    district_rows.append(
        {
            "district": int(district),
            "candidates": len(district_data),
            "pearson_r": r,
            "r_squared": r2,
        }
    )


district_summary = pd.DataFrame(
    district_rows
)


display(
    district_summary.round(
        {
            "pearson_r": 3,
            "r_squared": 3,
        }
    )
)


In [ ]:
fig_districts = px.bar(
    district_summary,
    x="district",
    y="pearson_r",
    text="pearson_r",
    hover_data={
        "candidates": True,
        "r_squared": ":.3f",
    },
    template="plotly_white",
    labels={
        "district": "District",
        "pearson_r": "Pearson correlation",
    },
    title=(
        "Fundraising → mentions (% of threshold) "
        "relationship across districts"
    ),
)


fig_districts.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
)


fig_districts.update_yaxes(
    range=[0, 1]
)


fig_districts.show()


## 6. District-by-district figures

These reproduce the weekly fundraising → support scatter/OLS plot for Districts 1–4, but with support measured as `% of STV threshold`.

Within a single district, the threshold is constant, so Pearson r and R² should match the raw-mentions version apart from floating-point rounding.


In [ ]:
def show_district(selected_district):
    # -----------------------------------------------------------
    # 1. Filter the data
    # -----------------------------------------------------------

    if selected_district == "All":
        data = plot_data.copy()
        title_label = "All districts"

    else:
        data = plot_data[
            plot_data["district"].eq(
                selected_district
            )
        ].copy()

        title_label = (
            f"District {selected_district}"
        )


    # -----------------------------------------------------------
    # 2. Calculate the relationship
    # -----------------------------------------------------------

    r, r2 = relationship_stats(
        data
    )


    # -----------------------------------------------------------
    # 3. Create the Plotly Express scatter
    # -----------------------------------------------------------

    fig = px.scatter(
        data,
        x="fundraising",
        y="mentions_pct_threshold",
        hover_name="canonical_candidate",
        hover_data={
            "district": True,
            "fundraising": ":$,.0f",
            "mentions": ":,.0f",
            "stv_threshold": ":,.0f",
            "mentions_pct_threshold": ":.1f",
            "first_place_votes": ":,.0f",
            "first_place_pct_threshold": ":.1f",
            "is_viable": True,
        },
        trendline="ols",
        template="plotly_white",
        labels={
            "fundraising": "Total fundraising ($)",
            "mentions_pct_threshold": (
                "Ballot mentions (% of STV threshold)"
            ),
        },
        title=(
            f"{title_label}: fundraising and mentions "
            "relative to threshold"
            f"<br><sup>Pearson r = {r:.2f} | "
            f"R² = {r2:.2f} | "
            f"n = {len(data)}</sup>"
        ),
    )


    fig.update_traces(
        marker={
            "size": 10,
            "opacity": 0.8,
        }
    )


    fig.add_hline(
        y=100,
        line_dash="dash",
        line_color="gray",
        annotation_text="100% threshold",
    )


    fig.show()


In [ ]:
for district in [1, 2, 3, 4]:
    show_district(district)


## 7. Optional interactive district explorer

Same explorer as the original weekly notebook, now using the normalized support outcome.


In [ ]:
district_dropdown = widgets.Dropdown(
    options=[
        ("All districts", "All"),
        ("District 1", 1),
        ("District 2", 2),
        ("District 3", 3),
        ("District 4", 4),
    ],
    value="All",
    description="District:",
)


district_output = widgets.Output()


def update_district(change):
    with district_output:
        clear_output(
            wait=True
        )

        show_district(
            change["new"]
        )


district_dropdown.observe(
    update_district,
    names="value",
)


display(
    district_dropdown,
    district_output,
)


# Show the initial graph.
with district_output:
    show_district(
        district_dropdown.value
    )


## 8. Find candidate pairs with similar fundraising but different normalized support

The same exploratory case-finding idea as Notebook 13b, now reported in threshold-relative support.


In [ ]:
MONEY_TOLERANCE = 0.10
SUPPORT_GAP = 0.50


pair_rows = []


for district in sorted(
    plot_data["district"].unique()
):
    district_data = (
        plot_data[
            plot_data["district"].eq(
                district
            )
        ]
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(district_data)
    ):
        for j in range(
            i + 1,
            len(district_data),
        ):
            candidate_a = district_data.iloc[i]
            candidate_b = district_data.iloc[j]


            larger_money = max(
                candidate_a["fundraising"],
                candidate_b["fundraising"],
            )

            money_gap = (
                abs(
                    candidate_a["fundraising"]
                    - candidate_b["fundraising"]
                )
                / larger_money
            )


            larger_support = max(
                candidate_a["mentions_pct_threshold"],
                candidate_b["mentions_pct_threshold"],
            )

            support_gap = (
                abs(
                    candidate_a["mentions_pct_threshold"]
                    - candidate_b["mentions_pct_threshold"]
                )
                / larger_support
            )


            if (
                money_gap <= MONEY_TOLERANCE
                and support_gap >= SUPPORT_GAP
            ):
                pair_rows.append(
                    {
                        "district": int(district),
                        "candidate_a": candidate_a[
                            "canonical_candidate"
                        ],
                        "candidate_b": candidate_b[
                            "canonical_candidate"
                        ],
                        "fundraising_a": candidate_a[
                            "fundraising"
                        ],
                        "fundraising_b": candidate_b[
                            "fundraising"
                        ],
                        "support_pct_a": candidate_a[
                            "mentions_pct_threshold"
                        ],
                        "support_pct_b": candidate_b[
                            "mentions_pct_threshold"
                        ],
                        "fundraising_gap_pct": (
                            money_gap * 100
                        ),
                        "support_gap_pct": (
                            support_gap * 100
                        ),
                    }
                )


interesting_pairs = pd.DataFrame(
    pair_rows
)


if not interesting_pairs.empty:
    interesting_pairs = (
        interesting_pairs
        .sort_values(
            [
                "district",
                "support_gap_pct",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


display(
    interesting_pairs.round(
        {
            "fundraising_a": 0,
            "fundraising_b": 0,
            "support_pct_a": 1,
            "support_pct_b": 1,
            "fundraising_gap_pct": 1,
            "support_gap_pct": 1,
        }
    )
)


## 9. Validation: raw mentions vs. threshold-normalized mentions within districts

Because every candidate in one district is divided by the same positive constant, the district-specific Pearson correlations should be the same.


In [ ]:
validation_rows = []


for district in [1, 2, 3, 4]:
    data = plot_data[
        plot_data["district"].eq(district)
    ].copy()

    raw_r = data["fundraising"].corr(
        data["mentions"]
    )

    pct_r = data["fundraising"].corr(
        data["mentions_pct_threshold"]
    )

    validation_rows.append(
        {
            "district": district,
            "raw_mentions_r": raw_r,
            "pct_threshold_r": pct_r,
            "difference": raw_r - pct_r,
        }
    )


validation = pd.DataFrame(
    validation_rows
)

display(
    validation.round(12)
)


## 10. Candidate values by district

In [ ]:
display(
    plot_data[
        [
            "district",
            "canonical_candidate",
            "fundraising",
            "mentions",
            "stv_threshold",
            "mentions_pct_threshold",
            "first_place_votes",
            "first_place_pct_threshold",
            "is_viable",
        ]
    ]
    .sort_values(
        [
            "district",
            "mentions_pct_threshold",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(
        {
            "fundraising": 0,
            "mentions": 0,
            "stv_threshold": 0,
            "mentions_pct_threshold": 1,
            "first_place_votes": 0,
            "first_place_pct_threshold": 1,
        }
    )
)
